In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

INPUT_PATH = "../../../data/phase2/labeled_signals.parquet"

# Phase 2 logistic regression baseline (from notebook 04)
LR_TRADES    = 444
LR_WIN_RATE  = 0.840
LR_BRIER     = 0.1440
LR_EV        = 0.5122

NADEX_WIN_PAYOUT = 0.80
NADEX_LOSS_COST  = 1.00

EXISTING_FEATURES = [
    "ema_ratio", "rsi_14", "macd_hist", "atr_14",
    "session_quality_enc", "direction_enc", "signal_valid_enc",
]
NEW_FEATURES = ["ema_20", "ema_50", "macd", "macd_signal", "instrument_enc"]
FEATURE_COLS = EXISTING_FEATURES + NEW_FEATURES
LABEL_COL = "label"
META_COLS = ["date", "s3_key", "fold", "split"]

In [ ]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived and encoded columns. Returns new DataFrame."""
    out = df.copy()
    out["ema_ratio"]           = out["ema_20"] / out["ema_50"]
    out["session_quality_enc"] = out["session_quality"].map({"high": 2, "medium": 1, "low": 0})
    out["direction_enc"]       = out["direction"].map({"buy": 1, "sell": -1, "none": 0})
    out["signal_valid_enc"]    = out["signal_valid"].astype(int)
    le = LabelEncoder()
    out["instrument_enc"]      = le.fit_transform(out["s3_key"])
    return out


def assign_walk_forward_folds(
    df: pd.DataFrame,
    train_months: int = 6,
    test_months: int = 1,
) -> pd.DataFrame:
    """
    Assign walk-forward fold metadata to each row.

    For each fold N:
      - test:  rows where date falls in [fold_start + train_months,
                                         fold_start + train_months + test_months)
      - train: rows where date falls in [fold_start, fold_start + train_months)
               AND the row is not already assigned to a test split

    Rows that don't fall into any fold's test window get fold=-1, split="unused".
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["fold"]  = -1
    df["split"] = "unused"

    min_date = df["date"].min().to_period("M")
    max_date = df["date"].max().to_period("M")
    month_period = df["date"].dt.to_period("M")

    fold_idx = 0
    cursor = min_date
    fold_windows = []
    while True:
        train_start = cursor
        train_end   = cursor + train_months
        test_start  = train_end
        test_end    = train_end + test_months
        if test_end > max_date + 1:
            break
        fold_windows.append((fold_idx, train_start, train_end, test_start, test_end))
        test_mask = (month_period >= test_start) & (month_period < test_end)
        df.loc[test_mask, "fold"]  = fold_idx
        df.loc[test_mask, "split"] = "test"
        fold_idx += 1
        cursor += test_months

    for fold_idx, train_start, train_end, test_start, test_end in fold_windows:
        train_mask = (
            (month_period >= train_start) &
            (month_period <  train_end) &
            (df["split"] != "test")
        )
        df.loc[train_mask, "fold"]  = fold_idx
        df.loc[train_mask, "split"] = "train"

    return df


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Drop NaN labels, encode features, assign folds. Returns model-ready DataFrame."""
    df = df[df["label"].notna()].copy()
    df = encode_features(df)
    df = assign_walk_forward_folds(df)
    keep = META_COLS + FEATURE_COLS + [LABEL_COL]
    return df[keep].reset_index(drop=True)

In [ ]:
raw = pd.read_parquet(INPUT_PATH)
prepped = prepare_features(raw)

print(f"Total rows after dropping direction=none: {len(prepped)}")
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"\nAny NaN in features: {prepped[FEATURE_COLS].isna().any().any()}")
print(f"\nLabel distribution:\n{prepped[LABEL_COL].value_counts()}")
print(f"\nFolds with train+test: {prepped[prepped['split']=='train']['fold'].nunique()}")